In [9]:
import os
from pathlib import Path
from uuid import uuid4

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, TextLoader, UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_chroma import Chroma

from src.md_reader import PreProcMd

#Set your ollama host to the port you started the server
OLLAMA_HOST = "127.0.0.1:40063"
os.environ["OLLAMA_HOST"] = "127.0.0.1:40063"
print(OLLAMA_HOST)

# ==========================================
EMBED_MODEL = "mxbai-embed-large:latest"
LLM_MODEL = "llama3.2:3b"
# ==========================================

127.0.0.1:40063


# Building Knowledge Base

## Loading document(s)
Here we'll load just a single document with classes from the [langchain-community](https://reference.langchain.com/python/langchain-community) document loader module. In the base example, we load a pdf file, but you may load pdf, txt, or markdown files. The loaders extract and load the files as [Document objects](https://reference.langchain.com/python/langchain-core/documents) defined in the [LangChain Core](https://reference.langchain.com/python/langchain-core) Python package.

In [2]:
def load_document(fpath: str):
    ext = os.path.splitext(fpath)[-1].lower()
    if ext == ".pdf":
        #mode: 'single' -> entire doc, (default) 'page' -> 1 doc/page
        loader = PyPDFLoader(fpath, mode="page")
        docs = loader.load()
    elif ext in [".md", ".markdown"]:
        repo_path = os.path.dirname(fpath)
        splits = PreProcMd(repo_path).run()
        target = os.path.abspath(fpath)
        docs = [
            d for d in splits
            if os.path.abspath(str(d.metadata.get("source", ""))) == target
        ]
    elif ext == ".txt":
        loader = TextLoader(fpath, encoding="utf-8")
        docs = loader.load()
    else:
        raise ValueError(f"Unsupported file type: {fpath}")
    print(f"Loaded {len(docs)} docs from {fpath}")
    return docs

In [3]:
docs_dir = '/mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs'
file_name = 'data_machine.pdf'
file_path = os.path.join(docs_dir, file_name) #path to all doc: /mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data
docs = load_document(file_path) #in 'page' mode number of docs here == number of pages in the actual document

Loaded 5 docs from /mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf


In [4]:
docs[0]

Document(metadata={'producer': 'pdfmake', 'creator': 'pdfmake', 'creationdate': '2025-10-24T14:13:13+00:00', 'source': '/mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='The Data Machine\n \nICER is excited to offer a new computational resource called the “Data Machine”. Many research \nareas are now faced with large amounts of data thanks in large part to the growth of available \ndatasets. Manipulating, analyzing, and visualizing  large amounts of data requires specialized \ncomputational resources that are not typically offered by more traditional high performance \ncomputing systems. Additionally, this “data explosion” is occurring primarily in fields where \nresearch computing has historically not been widely used.\n \nTo meet these hardware and educational needs, ICER is developing the Data Machine and \nassociated outreach and training programs. Though the machine is not yet available to t

## Splitting & Chunking Docs
We'll now take the loaded docs (list of Document objects), and split each of them into smaller chunks using the [`RecursiveCharacterTextSplitter`](https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter) from the [LangChain Text Splitters](https://reference.langchain.com/python/langchain-text-splitters) Python package. This applies splitting recursively based on typical splitting characters (e.g., ["\n\n", "\n", " ", ""]). The method first splits by paragraphs with "\n\n" and further splits each of those with "\n" (then space and empty string). Once the doc has been fully split, it then merges the split text to define chunks with maximum length determined by our set chunk size. 

The chunk size is essentially the maximum (and targeted) size of a chunk determined by length (len function by default). The chunk overlap is also a targeted value. Overlap helps maintain context when divided between chunks.

>Note the resulting chunk sizes and overlaps are often not exact, as these methods look for clean text boundaries (e.g., paragraph, line, word).  

In [5]:
def conditional_split(docs: list[Document], chunk_size: int, chunk_overlap: int) -> list[Document]:
    """
    Conditionally apply chunking only to documents longer than chunk_size.
    Docs <= chunk_size pass through unchanged (1 chunk).
    Docs > chunk_size are split using RecursiveCharacterTextSplitter with overlap.

    Params:
        docs: Input list of Documents (typically markdown will be already structurally split).
        chunk_size: Target max characters per chunk.
        chunk_overlap: Number of overlapping characters between consecutive chunks.

    Returns:
        A list of Documents where longer inputs are expanded into multiple chunks.
    """
    splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap,add_start_index=True, separators=None, ) #default splitting characters ["\n\n", "\n", " ", ""]
    # splitter=CharacterTextSplitter(chunk_size=chunk_size,chunk_overlap=chunk_overlap,add_start_index=True,)
    out = []
    for i in docs:
        if len(i.page_content) <= chunk_size:
            out.append(i)  
        else:
            out.extend(splitter.split_documents([i]))  
    return out

In [6]:
# text_splitter = RecursiveCharacterTextSplitter(
#         chunk_size=1000, chunk_overlap=200, add_start_index=True
#     )
splits = conditional_split(docs, chunk_size=700, chunk_overlap=100)
#splits = text_splitter.split_documents(all_docs)
print(f"Split into {len(splits)} chunks.")

Split into 21 chunks.


In [8]:
# [len(s.page_content) for s in splits]

In [62]:
splits[0].page_content[-150:]

'ere \nresearch computing has historically not been widely used.\n \nTo meet these hardware and educational needs, ICER is developing the Data Machine and'

In [63]:
splits[1].page_content[:150]

'To meet these hardware and educational needs, ICER is developing the Data Machine and \nassociated outreach and training programs. Though the machine i'

## Document Embedding
Next, we will embed the documents and put them in a vector store collection. We'll use the [LangChain Ollama embedding integration](https://reference.langchain.com/python/langchain-ollama/embeddings/OllamaEmbeddings) to embed the docs with a local embedding model from our Ollama common model space. Here, we're using a [Chroma](https://docs.trychroma.com/docs/overview/introduction) vector store with the [LangChain Chroma integration](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma).

In [101]:
# embeddings = OllamaEmbeddings(model=EMBED_MODEL)
# vector_store = InMemoryVectorStore(embeddings)
# vector_store.add_documents(splits)
# print(f"{len(vector_store.store)} docs in vector store, one for each chunk.")

### Creating the vector store
The vector store is then created with name for our docs collection, and the `OllamaEmbeddings` instance to define/configure the embedding function. Note the `OllamaEmbeddings` class is used to store configuration data of the selected embedding model, and implements the embedding function using the selected embedding model. 

> In this case the vector store is setup and running in-memory, not on disk/external db.

> Run vector_store._client.delete_collection(collection_name) to remove collection from chroma vector store

In [64]:
collection_name = "icer_docs_collection"

embeddings = OllamaEmbeddings(model=EMBED_MODEL)
vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
)

#vector_store._client.delete_collection("icer_docs_collection")

Now we add the documents to the vector store, providing unique ids to each document. The document embeddings are included in this step.

In [65]:
%%time

uuids = [str(uuid4()) for _ in range(len(splits))]
vector_store.add_documents(documents=splits, ids=uuids) # add_documents applies the embedding function to the documents (splits)
collection = vector_store._collection
n_docs_stored = collection.count()
print(f"{n_docs_stored} docs in vector store, one for each chunk.")

21 docs in vector store, one for each chunk.
CPU times: user 32.2 ms, sys: 27.5 ms, total: 59.7 ms
Wall time: 25.1 s


In [66]:
vs_embedding_info = vector_store._collection.get(include=["embeddings"])
doc_embeddings = vs_embedding_info['embeddings']

print(f"{doc_embeddings.shape[0]} doc embeddings of size {doc_embeddings.shape[1]} in vector store: \n\n", doc_embeddings)

21 doc embeddings of size 1024 in vector store: 

 [[ 0.02535876  0.00185163 -0.02738773 ... -0.01723007 -0.00513023
   0.03616323]
 [ 0.05360975  0.0255984  -0.051535   ... -0.01254294 -0.01470418
   0.0433056 ]
 [ 0.02566486 -0.00202891  0.01027462 ...  0.00027519 -0.01499113
   0.06011929]
 ...
 [ 0.06266174 -0.01104502 -0.00123992 ... -0.07037085 -0.01103648
   0.01724077]
 [ 0.05125768 -0.0210232  -0.02039827 ... -0.03477856 -0.00246366
   0.02483794]
 [ 0.02858585 -0.00039287 -0.02574478 ... -0.0227232   0.00218294
   0.03089899]]


In [67]:
print(vs_embedding_info)

{'ids': ['a01b4e66-456a-432a-a333-dae7ec9308ea', 'cc78563b-862d-4178-853e-8b8c2ca28327', 'd25e2885-e399-4fec-9c37-258b377191a2', 'fa0fef9d-b5ed-48b2-bc55-81a81b62d7ad', 'bb320f6c-7457-4ba4-a408-1ffc47d17cb9', 'e25edf39-1fef-48ee-ac47-e722d89386a9', 'ab9870b1-1c9b-4f11-a591-7126a4e92454', '10b2c6b8-62e7-494f-ab6b-afedc4357afd', '6aedeef4-a9af-484d-a31b-bf531e3c37e6', 'd03fbbf6-d952-4b89-bb30-9cea89fee7e4', 'c33f8f8a-4f58-4cb6-aa89-3f8ae4515a2d', '7f104bb3-824e-41e9-8869-f5f99b224be4', '07422caf-3b9e-45eb-85a0-9451b0fd5a7d', 'c5b78692-662e-4266-9f37-73e01423db55', '41fb19f5-cea6-4c24-86da-d553649e7770', '8372df0a-1f7a-40e3-b266-0b750f11058e', '967353a7-19f9-4982-b4a3-c64b1f51c1e2', '0381ce91-edeb-4d16-8d21-1f02d9d4c51e', 'da44a89b-312a-42d1-8207-5b5b2826226f', '52abfb16-cd8e-4a58-8088-2d87ec66f323', '23021556-4ddc-41ce-8fad-25e9713a1078'], 'embeddings': array([[ 0.02535876,  0.00185163, -0.02738773, ..., -0.01723007,
        -0.00513023,  0.03616323],
       [ 0.05360975,  0.0255984 , -0

In [80]:
# vector_store._collection.get(include=["embeddings"]) #include ["documents", "metadatas", "embeddings"]

Here we run a similarity search for a simple query. The query is embedded and checked against the vector store embeddings, selecting the top k docs by similarity (or distance). Another method is using [`similarity_search_with_relevance_scores`](https://reference.langchain.com/python/langchain-core/vectorstores/base/VectorStore/similarity_search_with_relevance_scores) which scores increasing with similarity.  

In [68]:
results = vector_store.similarity_search_with_score(
    "What type of GPUs are on The Data Machine?",
    k=5,
    filter={"source": file_path},
)
for res, score in results:
    print(f"* [similarity={score:3f}] {res.metadata} \n")
    print(res.page_content + "\n")

* [similarity=0.465068] {'source': '/mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf', 'creationdate': '2025-10-24T14:13:13+00:00', 'page': 1, 'producer': 'pdfmake', 'creator': 'pdfmake', 'page_label': '2', 'total_pages': 5, 'start_index': 2216} 

transferring large volumes of data. The Data Machine will have a total of 8 nodes, 4 focused on 
CPU-intensive jobs and 4 for jobs requiring GPUS.
 
The Data Machine will be connected to the existing HPCC file systems and compute resources. 
Though the CPU and GPU hardware is similar to what is offered by the current amd21 and 
amd22 clusters, what sets the Data Machine apart is the amount of memory available to CPU

* [similarity=0.482812] {'creationdate': '2025-10-24T14:13:13+00:00', 'producer': 'pdfmake', 'source': '/mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf', 'start_index': 531, 'total_pages': 5, 'creator': 'pdfmake', 'page_label': '3', 'page': 2} 

units, making them ideal for m

In [69]:
results = vector_store.similarity_search_with_relevance_scores(
    "What type of GPUs are on The Data Machine?",
    k=5,
    filter={"source": file_path},
)
for res, score in results:
    print(f"* [similarity={score:3f}] {res.metadata} \n")
    print(res.page_content + "\n")

* [similarity=0.671147] {'creationdate': '2025-10-24T14:13:13+00:00', 'producer': 'pdfmake', 'page': 1, 'total_pages': 5, 'source': '/mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf', 'creator': 'pdfmake', 'page_label': '2', 'start_index': 2216} 

transferring large volumes of data. The Data Machine will have a total of 8 nodes, 4 focused on 
CPU-intensive jobs and 4 for jobs requiring GPUS.
 
The Data Machine will be connected to the existing HPCC file systems and compute resources. 
Though the CPU and GPU hardware is similar to what is offered by the current amd21 and 
amd22 clusters, what sets the Data Machine apart is the amount of memory available to CPU

* [similarity=0.658600] {'page': 2, 'source': '/mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf', 'start_index': 531, 'total_pages': 5, 'producer': 'pdfmake', 'creator': 'pdfmake', 'creationdate': '2025-10-24T14:13:13+00:00', 'page_label': '3'} 

units, making them ideal for m

## Building Knowledge Base
We'll now extend document embedding to a set of docs (pdfs in this example).

In [70]:
docs_repo = Path(docs_dir)
file_paths = list(docs_repo.glob("*.pdf"))
all_docs = []
for path in file_paths:
    all_docs.extend(load_document(path))

splits = conditional_split(all_docs, chunk_size=700, chunk_overlap=100)
print(f"Split into {len(splits)} chunks.")

Loaded 5 docs from /mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf
Loaded 3 docs from /mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/requesting_gpus.pdf
Loaded 3 docs from /mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/cluster_resources.pdf
Loaded 5 docs from /mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/llms_hpcc.pdf
Loaded 2 docs from /mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/gpu_resources.pdf
Loaded 2 docs from /mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/user_data_policies.pdf
Loaded 1 docs from /mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/dev_nodes.pdf
Loaded 3 docs from /mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/job_policies.pdf
Loaded 3 docs from /mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/interactive_job.pdf
Loaded 12 docs from /mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/faq.pdf
Split into 139 chunks.


In [71]:
vector_store = Chroma(
    collection_name="icer_docs_collection_full",
    embedding_function=embeddings,
    persist_directory="./chroma_icer_docs_db",
)
print(vector_store._collection.get())

{'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}


In [72]:
%%time

uuids = [str(uuid4()) for _ in range(len(splits))]
vector_store.add_documents(documents=splits, ids=uuids)
collection = vector_store._collection
n_docs_stored = collection.count()
print(f"{n_docs_stored} docs in vector store, one for each chunk.")

139 docs in vector store, one for each chunk.
CPU times: user 134 ms, sys: 38.3 ms, total: 172 ms
Wall time: 2min 10s


You can now query against a larger set of documents. As you can see, it may be nice to not have to run the embeddings each time you want to query against your documents, and instead query against the built and exisisting vector store. 

In [76]:
results = vector_store.similarity_search_with_score(
    "What type of GPUs are on The Data Machine?",
    k=5,
    filter={"source": file_path},
)
for res, score in results:
    print(f"* [similarity={score:3f}] {res.metadata} \n")
    print(res.page_content + "\n")

* [similarity=0.465068] {'producer': 'pdfmake', 'creator': 'pdfmake', 'page_label': '2', 'creationdate': '2025-10-24T14:13:13+00:00', 'total_pages': 5, 'start_index': 2216, 'page': 1, 'source': '/mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf'} 

transferring large volumes of data. The Data Machine will have a total of 8 nodes, 4 focused on 
CPU-intensive jobs and 4 for jobs requiring GPUS.
 
The Data Machine will be connected to the existing HPCC file systems and compute resources. 
Though the CPU and GPU hardware is similar to what is offered by the current amd21 and 
amd22 clusters, what sets the Data Machine apart is the amount of memory available to CPU

* [similarity=0.482812] {'creator': 'pdfmake', 'creationdate': '2025-10-24T14:13:13+00:00', 'producer': 'pdfmake', 'page': 2, 'start_index': 531, 'page_label': '3', 'source': '/mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs/data_machine.pdf', 'total_pages': 5} 

units, making them ideal for m

Now we'll delete the vector store, and re-set to the persistent vector store db. We should see a collection already there. Now we do not need to re-apply the embedding function to our reference docs.

In [6]:
del vector_store

In [7]:
vector_store = Chroma(
    collection_name="icer_docs_collection_full",
    embedding_function=embeddings,
    persist_directory="./chroma_icer_docs_db",
)
print(vector_store._collection.get(include=["embeddings"]))

{'ids': ['c8b683ec-60ba-4481-94bf-0a545a5a0db1', '74c54274-76a1-4d9a-b333-ac4ba3794d1b', '4af326a0-78bf-42e8-86d8-5fb1a59d5497', '0d5acdd4-601b-48e7-92b6-0cbe5a8f95f3', '322c2740-ab15-44e3-bbbd-c188966f6d34', '2cc26dba-6e21-4820-82c9-5c6123320f8e', '3d01351e-6c03-4cf4-8ea4-85deb743a75f', 'ee6258c5-4121-46f0-ab4b-35b1c4e98afc', '3d5a202a-6403-4e0e-96e1-00f6c4049a2f', '5206ed3e-f3c2-4bab-9c60-9f6802651204', '351c85cd-4282-4a77-ba7f-1ef2c98dcafa', '1b7a486d-702d-438d-9efc-2ae10467366c', 'c0c6da66-9234-4063-a934-be0e5480d3b0', 'fcfdb3ea-e37e-4c53-a120-939347b1f1cd', '77815bae-d207-4bb2-a588-5fb2892c9cdb', '2606298b-c8dc-4c38-82eb-b348440a04bb', '862554aa-2675-4ecc-851a-986bf8a7274d', '1dd60f18-9a1f-4e9a-92e2-8dca213e8b76', '4d93d9c5-93c2-40d1-a021-c15152f253dd', '64429005-76f4-493a-bb20-f408d15a8aae', '2479de77-5d21-433f-8e12-cc6399849adf', '9c862f2c-5e00-44d8-adae-e268bfa399c3', '8ba5d0d2-8109-4990-844e-976afda50b4c', '7ef30b50-710c-475b-ada6-20311ba39080', 'e81de12a-fa53-4126-9ae9-8bf4ab

We may want to 

In [ ]:
file_path = '/mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs_2/LabNotebook_Update_HowtorunRAGinOpenWebUI.pdf'
docs = load_document(file_path) #in 'page' mode number of docs here == number of pages in the actual document

Loaded 1 docs from /mnt/ufs18/rs-007/HPCC_AI_Workshop/use-case/data/icer_docs_mkdwn/2025-03-05_LabNotebook_How_to_run_OpenWebUI_RAG.md


In [87]:
new_splits = conditional_split(docs, chunk_size=700, chunk_overlap=100)
#splits = text_splitter.split_documents(all_docs)
print(f"Split into {len(new_splits)} chunks.")

Split into 8 chunks.


In [8]:
# uuids = [str(uuid4()) for _ in range(len(new_splits))]
# vector_store.add_documents(documents=new_splits, ids=uuids) # add_documents applies the embedding function to the new documents (new splits)